# 🏠 Projet Immobilier : Analyse & Consolidation des Données

**Auteur :** Daryl  
**Dernière mise à jour :** Septembre 2026  
**Objectif :** Exploration, nettoyage, consolidation et imputation des différentes sources de données du marché immobilier français.

---

### 📑 Table des Matières
1. [Configuration & Initialisation](#1.-Configuration-&-Initialisation)
2. [Données Macro-économiques (National / Temporel)](#2.-Donn%C3%A9es-Macro-%C3%A9conomiques-(National-/-Temporel))
   - 2.1 Flux d'emprunts
   - 2.2 Indice de Référence des Loyers (IRL)
   - 2.3 Consolidation Macro (`df_macro`)
3. [Données Territoriales & Locales (Communes / Annuel)](#3.-Donn%C3%A9es-Territoriales-&-Locales-(Communes-/-Annuel))
   - 3.1 Loyers au m²
   - 3.2 Foyers fiscaux & Revenus
   - 3.3 Parc immobilier & Logements vacants
   - 3.4 Consolidation Territoriale (`df_territorial`)
   - 3.5 Imputation globale de toutes les valeurs manquantes
4. [Exploration des Transactions Immobilières](#4.-Exploration-des-Transactions-Immobili%C3%A8res)
5. [Synthèse & Prochaines Étapes](#5.-Synth%C3%A8se-&-Prochaines-%C3%89tapes)

## 1. Configuration & Initialisation

In [28]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Paramètres d'affichage
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)

# Chemins des dossiers
DATA_DIR = Path('data')

## 2. Données Macro-économiques (National / Temporel)
Cette section regroupe les indicateurs globaux au niveau national : flux des nouveaux emprunts et indice IRL.

### 2.1 Flux des Nouveaux Emprunts (Mensuel)

In [29]:
# Chargement et conversion de la date en période mensuelle
data_emprunt = pd.read_csv(DATA_DIR / 'flux_nouveaux_emprunts.csv')
data_emprunt['date'] = pd.to_datetime(data_emprunt['date'], format='%Y-%m').dt.to_period('M')

print(f"Dimensions des emprunts : {data_emprunt.shape}")
display(data_emprunt.head())

Dimensions des emprunts : (170, 2)


,date,emprunts_M€
0,2024-07,16442
1,2024-06,13017
2,2024-05,10709
3,2024-04,10817
4,2024-03,9751


### 2.2 Indice de Référence des Loyers - IRL (Trimestriel)

In [30]:
# Chargement et conversion de la date en période mensuelle pour harmonisation
data_ref_loyers = pd.read_csv(DATA_DIR / 'indice_reference_loyers.csv')
data_ref_loyers['date'] = pd.to_datetime(data_ref_loyers['quarter'], format='%Y-%m-%d').dt.to_period('M')

print(f"Dimensions de l'IRL : {data_ref_loyers.shape}")
display(data_ref_loyers.head())

Dimensions de l'IRL : (87, 3)


,quarter,IRL,date
0,2024-04-01,145.17,2024-04
1,2024-01-01,143.46,2024-01
2,2023-10-01,142.06,2023-10
3,2023-07-01,141.03,2023-07
4,2023-04-01,140.59,2023-04


### 2.3 Consolidation Macro-économique (`df_macro`)
Fusion des emprunts et de l'IRL sur la colonne `date`. Utilisation de `bfill().ffill()` pour propager l'indice trimestriel aux mois intermédiaires et éliminer tous les NaN.

In [31]:
df_macro = pd.merge(
    data_emprunt,
    data_ref_loyers[['date', 'IRL']],
    on='date',
    how='left'
)

# Propagation de la valeur trimestrielle de l'IRL aux mois manquants
df_macro['IRL'] = df_macro['IRL'].bfill().ffill()

print(f"Dimensions de df_macro : {df_macro.shape}")
print(f"Valeurs manquantes dans df_macro : {df_macro.isna().sum().to_dict()}")
display(df_macro.head(10))

Dimensions de df_macro : (170, 3)
Valeurs manquantes dans df_macro : {'date': 0, 'emprunts_M€': 0, 'IRL': 0}


,date,emprunts_M€,IRL
0,2024-07,16442,145.17
1,2024-06,13017,145.17
2,2024-05,10709,145.17
3,2024-04,10817,145.17
4,2024-03,9751,143.46
5,2024-02,8811,143.46
6,2024-01,9487,143.46
7,2023-12,11993,142.06
8,2023-11,10947,142.06
9,2023-10,11866,142.06


## 3. Données Territoriales & Locales (Communes / Annuel)
Cette section regroupe les données par ville : niveaux de loyers, profil fiscal des ménages, et vacance des logements.

### 3.1 Loyers d'appartement & maison par commune

In [32]:
data_loyers = pd.read_csv(DATA_DIR / 'loyers.csv')

# Standardisation des types de colonnes clés
data_loyers['date'] = pd.to_datetime(data_loyers['date'], format='%Y').dt.to_period('Y')
data_loyers['departement'] = data_loyers['departement'].astype(str).str.zfill(2)

print(f"Dimensions des loyers : {data_loyers.shape}")
display(data_loyers.head())

Dimensions des loyers : (105391, 6)


,departement,id_ville,ville,date,loyer_m2_appartement,loyer_m2_maison
0,01,1,L'Abergement-Clémenciat,2018,9.372335,8.191701
1,01,2,L'Abergement-de-Varey,2018,8.635552,7.441199
2,01,4,Ambérieu-en-Bugey,2018,10.074507,8.411184
3,01,5,Ambérieux-en-Dombes,2018,9.372335,9.039551
4,01,6,Ambléon,2018,8.966955,8.058265


### 3.2 Foyers fiscaux & Revenus par commune

In [33]:
data_fiscaux = pd.read_csv(DATA_DIR / 'foyers_fiscaux.csv')

# Standardisation des types de colonnes clés
data_fiscaux['date'] = pd.to_datetime(data_fiscaux['date'], format='%Y').dt.to_period('Y')
data_fiscaux['departement'] = data_fiscaux['departement'].astype(str).str.zfill(2)

print(f"Dimensions des foyers fiscaux : {data_fiscaux.shape}")
display(data_fiscaux.head())

Dimensions des foyers fiscaux : (315542, 15)


,date,departement,id_ville,ville,n_foyers_fiscaux,revenu_fiscal_moyen,montant_impot_moyen,n_foyers_0k_10k,n_foyers_10k_12k,n_foyers_12k_15k,n_foyers_15k_20k,n_foyers_20k_30k,n_foyers_30k_50k,n_foyers_50k_100k,n_foyers_100k_plus
0,2022,01,1,L'Abergement-Clémenciat,466.0,35291.10,1718.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022,01,2,L'Abergement-de-Varey,143.0,34788.93,1728.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022,01,4,Ambérieu-en-Bugey,9276.0,26612.43,1251.87,1846.0,450.0,705.0,1630.0,1971.0,1668.0,894.0,112.0
3,2022,01,5,Ambérieux-en-Dombes,1073.0,34712.21,2078.79,100.0,30.0,70.0,177.0,234.0,280.0,158.0,24.0
4,2022,01,6,Ambléon,78.0,26655.58,1347.87,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.3 Parc immobilier & Logements vacants

In [34]:
data_parc = pd.read_csv(DATA_DIR / 'parc_immobilier.csv')

# Standardisation des types
data_parc['date'] = pd.to_datetime(data_parc['date'], format='%Y').dt.to_period('Y')
data_parc['departement'] = data_parc['departement'].astype(str).str.zfill(2)

print(f"Dimensions du parc immobilier : {data_parc.shape}")
display(data_parc.head())

Dimensions du parc immobilier : (48627, 6)


,date,departement,id_ville,ville,n_logements,n_logements_vacants
0,2019,01,4,Ambérieu-en-Bugey,NaN,565.0
1,2019,01,7,Ambronay,NaN,78.0
2,2019,01,10,Anglefort,NaN,64.0
3,2019,01,14,Arbent,NaN,86.0
4,2019,01,15,Arboys en Bugey,NaN,33.0


### 3.4 Consolidation Territoriale (`df_territorial`)
Fusion des loyers, de la fiscalité et du parc immobilier par identifiant de commune (`id_ville`, `departement`, `date`).

In [35]:
# 1. Jointure Loyers + Foyers Fiscaux
df_territorial = pd.merge(
    data_loyers,
    data_fiscaux,
    on=['departement', 'id_ville', 'date'],
    how='inner',
    suffixes=('', '_fiscaux')
)

# 2. Ajout de la métrique n_logements_vacants issue du parc immobilier
df_territorial = pd.merge(
    df_territorial,
    data_parc[['departement', 'id_ville', 'n_logements_vacants']].drop_duplicates(subset=['departement', 'id_ville']),
    on=['departement', 'id_ville'],
    how='left'
)

print(f"Dimensions de df_territorial : {df_territorial.shape}")
display(df_territorial.head())

Dimensions de df_territorial : (69796, 19)


,departement,id_ville,ville,date,loyer_m2_appartement,loyer_m2_maison,ville_fiscaux,n_foyers_fiscaux,revenu_fiscal_moyen,montant_impot_moyen,n_foyers_0k_10k,n_foyers_10k_12k,n_foyers_12k_15k,n_foyers_15k_20k,n_foyers_20k_30k,n_foyers_30k_50k,n_foyers_50k_100k,n_foyers_100k_plus,n_logements_vacants
0,01,1,L'Abergement-Clémenciat,2018,9.372335,8.191701,L'Abergement-Clémenciat,433.0,30733.42,1633.95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01,2,L'Abergement-de-Varey,2018,8.635552,7.441199,L'Abergement-de-Varey,132.0,34259.75,1660.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01,4,Ambérieu-en-Bugey,2018,10.074507,8.411184,Ambérieu-en-Bugey,8391.0,26238.04,1594.17,1696.0,500.0,846.0,1508.0,1619.0,1445.0,685.0,92.0,565.0
3,01,5,Ambérieux-en-Dombes,2018,9.372335,9.039551,Ambérieux-en-Dombes,926.0,30834.32,1869.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01,6,Ambléon,2018,8.966955,8.058265,Ambléon,60.0,28507.40,1622.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.5 Imputation Globale de Toutes les Valeurs Manquantes dans `df_territorial`

Pipeline complet d'imputation selon la nature des variables :
1. **`n_foyers_fiscaux`** : Médiane départementale, puis globale.
2. **`revenu_fiscal_moyen` & `montant_impot_moyen`** : Médiane départementale, puis globale.
3. **Tranches de revenus (`n_foyers_0k_10k` à `100k_plus`)** : Imputation proportionnelle basée sur le ratio national de chaque tranche appliqué à `n_foyers_fiscaux`.
4. **`n_logements_vacants`** : Estimation par le taux de vacance médian départemental, puis médiane départementale/globale.

In [36]:
print("=== Bilan des valeurs manquantes AVANT imputation ===")
print(df_territorial.isna().sum()[df_territorial.isna().sum() > 0])

# 1. Imputation de n_foyers_fiscaux (Médiane départementale puis globale)
df_territorial['n_foyers_fiscaux'] = df_territorial['n_foyers_fiscaux'].fillna(
    df_territorial.groupby('departement')['n_foyers_fiscaux'].transform('median')
).fillna(df_territorial['n_foyers_fiscaux'].median())

# 2. Imputation du revenu fiscal moyen et du montant d'impôt moyen
for col in ['revenu_fiscal_moyen', 'montant_impot_moyen']:
    df_territorial[col] = df_territorial[col].fillna(
        df_territorial.groupby('departement')[col].transform('median')
    ).fillna(df_territorial[col].median())

# 3. Imputation des tranches de revenus par distribution proportionnelle au total de foyers
tranche_cols = [
    'n_foyers_0k_10k', 'n_foyers_10k_12k', 'n_foyers_12k_15k', 'n_foyers_15k_20k',
    'n_foyers_20k_30k', 'n_foyers_30k_50k', 'n_foyers_50k_100k', 'n_foyers_100k_plus'
]
for col in tranche_cols:
    ratio = (df_territorial[col] / df_territorial['n_foyers_fiscaux']).median()
    df_territorial[col] = df_territorial[col].fillna(np.round(df_territorial['n_foyers_fiscaux'] * ratio))

# 4. Imputation de n_logements_vacants (médiane départementale puis globale)
df_territorial['n_logements_vacants'] = df_territorial['n_logements_vacants'].fillna(
    df_territorial.groupby('departement')['n_logements_vacants'].transform('median')
).fillna(df_territorial['n_logements_vacants'].median())

print("\n=== Bilan des valeurs manquantes APRÈS imputation ===")
print(f"Total des NaN restants dans df_territorial : {df_territorial.isna().sum().sum()}")
display(df_territorial.head(10))

=== Bilan des valeurs manquantes AVANT imputation ===
n_foyers_fiscaux          97
revenu_fiscal_moyen      183
montant_impot_moyen      183
n_foyers_0k_10k        58069
n_foyers_10k_12k       58436
n_foyers_12k_15k       58070
n_foyers_15k_20k       58069
n_foyers_20k_30k       58069
n_foyers_30k_50k       58069
n_foyers_50k_100k      58696
n_foyers_100k_plus     59049
n_logements_vacants    42101
dtype: int64

=== Bilan des valeurs manquantes APRÈS imputation ===
Total des NaN restants dans df_territorial : 0


,departement,id_ville,ville,date,loyer_m2_appartement,loyer_m2_maison,ville_fiscaux,n_foyers_fiscaux,revenu_fiscal_moyen,montant_impot_moyen,n_foyers_0k_10k,n_foyers_10k_12k,n_foyers_12k_15k,n_foyers_15k_20k,n_foyers_20k_30k,n_foyers_30k_50k,n_foyers_50k_100k,n_foyers_100k_plus,n_logements_vacants
0,01,1,L'Abergement-Clémenciat,2018,9.372335,8.191701,L'Abergement-Clémenciat,433.0,30733.42,1633.95,72.0,21.0,34.0,71.0,85.0,89.0,45.0,7.0,63.0
1,01,2,L'Abergement-de-Varey,2018,8.635552,7.441199,L'Abergement-de-Varey,132.0,34259.75,1660.04,22.0,6.0,10.0,21.0,26.0,27.0,14.0,2.0,63.0
2,01,4,Ambérieu-en-Bugey,2018,10.074507,8.411184,Ambérieu-en-Bugey,8391.0,26238.04,1594.17,1696.0,500.0,846.0,1508.0,1619.0,1445.0,685.0,92.0,565.0
3,01,5,Ambérieux-en-Dombes,2018,9.372335,9.039551,Ambérieux-en-Dombes,926.0,30834.32,1869.20,155.0,45.0,72.0,151.0,182.0,190.0,97.0,15.0,63.0
4,01,6,Ambléon,2018,8.966955,8.058265,Ambléon,60.0,28507.40,1622.32,10.0,3.0,5.0,10.0,12.0,12.0,6.0,1.0,63.0
5,01,7,Ambronay,2018,9.039236,7.441199,Ambronay,1432.0,28569.74,1332.40,192.0,65.0,129.0,245.0,267.0,320.0,203.0,11.0,78.0
6,01,8,Ambutrix,2018,8.635552,8.022279,Ambutrix,431.0,30642.17,1775.95,72.0,21.0,33.0,70.0,85.0,88.0,45.0,7.0,63.0
7,01,9,Andert-et-Condon,2018,8.899091,7.641314,Andert-et-Condon,202.0,35797.41,2752.32,34.0,10.0,16.0,33.0,40.0,41.0,21.0,3.0,63.0
8,01,10,Anglefort,2018,11.172664,10.538127,Anglefort,588.0,27397.17,834.79,98.0,29.0,46.0,96.0,115.0,121.0,61.0,9.0,64.0
9,01,11,Apremont,2018,8.931485,7.441199,Apremont,209.0,29457.31,1906.79,35.0,10.0,16.0,34.0,41.0,43.0,22.0,3.0,63.0


## 4. Exploration des Transactions Immobilières
Aperçu de l'échantillon des transactions de ventes immobilières (`transactions_sample.csv`).

In [37]:
data_transactions = pd.read_csv(DATA_DIR / 'transactions_sample.csv')
print(f"Dimensions de transactions_sample : {data_transactions.shape}")
display(data_transactions.head())

Dimensions de transactions_sample : (100, 20)


,id_transaction,date_transaction,prix,departement,id_ville,ville,code_postal,adresse,type_batiment,vefa,n_pieces,surface_habitable,id_parcelle_cadastre,latitude,longitude,surface_dependances,surface_locaux_industriels,surface_terrains_agricoles,surface_terrains_sols,surface_terrains_nature
0,10160888,2015-07-22,222500.0,63,247,MUROL,63790,5148 COMBE,Maison,False,4,123,63247000ZO0165,45.572973,2.949976,{},{},{},{2387},{}
1,10319766,2024-06-18,218640.0,73,15,LES ALLUES,73550,45 RUE DU GRAND COEUR,Appartement,False,1,23,73015000AB0347,45.398740,6.567602,{0},{},{},{},{}
2,11545562,2020-07-23,254950.0,77,316,MORET-LOING-ET-ORVANNE,77250,1 RUE DE LA CROIX BLANCHE,Maison,False,6,124,773161700B0305,48.333762,2.780830,{},{},{},{815},{}
3,13891173,2022-10-03,380000.0,94,81,VITRY-SUR-SEINE,94400,11 VOIE VICTOR MASSE,Maison,False,3,93,94081000AK0189,48.796988,2.375558,{},{},{},{208},{}
4,11794772,2018-07-12,258000.0,77,284,MEAUX,77100,14 RUE PIERRE BONTEMPS,Maison,False,5,96,77284000AW0090,48.948816,2.891734,{},{},{},{},{}


## 5. Synthèse & Prochaines Étapes

### 📊 Datasets Consolidés & Nettoyés :
1. **`df_macro`** : Séries chronologiques nationales (taux d'emprunt, IRL mensuel consolidé, **0 NaN**).
2. **`df_territorial`** : Référentiel complet par commune (loyers moyens appartement/maison, fiscalité complète, logements vacants, **0 NaN**).
3. **`data_transactions`** : Échantillon des transactions immobilières individuelles.

### 🚀 Prochaines étapes suggérées :
- Analyses statistiques descriptives & corrélations (ex: loyers vs revenus fiscaux).
- Visualisations graphiques (`matplotlib` / `seaborn` / `plotly`).
- Modélisation prédictive ou clustering géographique des communes.